# 第8章 Element-Wise：逐元素算子操作手册

**仓库定位器**：本 notebook 位于 `hello-gpu/notebooks/part2-kernels/chapter8.ipynb`，对应源码目录 `hello-gpu/code/part2-kernels/chapter8/`，文档位于 `hello-gpu/docs/part2-kernels/chapter8/index.md`。

---

## Goal

本章以 Vector Add 为例，分别用 HIP 和 Triton 实现逐元素算子，验证正确性并测量性能。你将学会：

**核心原则**：correctness before performance（正确性优先于性能）

- 理解 Element-Wise 算子的数据依赖特征
- 掌握 HIP 线程级范式与 Triton 分块范式的编程差异
- 运行边界正确性检查（N=31, 32, 33, 1027 等）
- 执行 GPU event 计时并计算逻辑有效带宽
- 分析 kernel trace 资源字段
- 理解受控访存实验的设计与结果

---

## Prerequisite

- 已完成第 4-7 章：GPU 环境、可信计时、kernel trace、Roofline 基础
- 熟悉 Python 与基础 C++
- 当前平台具备可用的 ROCm 与 PyTorch ROCm；实际版本由运行时记录，不作为硬编码门槛
- 理解 FP32 浮点运算与内存带宽概念

---

## 平台

**参考平台**：AMD Radeon RX 9070 XT (gfx1201) + ROCm 7.13 + 原生 Ubuntu 24.04.4 LTS。该组合只描述已提交 reference，不是当前平台的版本门。

**其他平台注意事项**：
- **gfx1100 / gfx1151**：可运行，但性能数据与 gfx1201 参考结果不直接比较
- **gfx1201**：本章性能数据均来自参考平台
- 跨平台运行时默认由 `rocminfo` 精确检测 GPU Agent `Name: gfx...`；仅在有意指定编译 target 时设置 `HELLO_GPU_ARCH`，不设置 HSA override。

---

## Parameter

### 主要参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `SIZE` | 16777216 | 输入向量长度（FP32 元素数） |
| `HIP_BLOCK` | 256 | HIP kernel 的 block size |
| `TRITON_BLOCK` | 1024 | Triton t1 的 BLOCK_SIZE（t0 固定为 256） |
| `WARMUP` | 10 | 预热迭代次数 |
| `REPEAT` | 50 | 正式计时迭代次数 |
| `SEED` | 20260716 | 输入生成随机种子 |
| `INDEPENDENT_RUNS` | 3 | 独立进程复跑次数 |
| `RUN_CH8_FORMAL` | `True` | Notebook 默认执行正确性、benchmark 与 HIP profiling |
| `TRITON_PROFILE_MODE` | 自动（`direct` 或 `skip`） | Python 环境自带配套 ROCm SDK（如 pip 安装的 torch ROCm 轮子）时默认 `direct`，并用环境自带 rocprofv3；否则默认 `skip`；可用环境变量显式覆盖 |
| `FORMAL_TIMEOUT_SECONDS` | 900 | `run_all.sh` 超时上限（秒） |
| `PROFILE_TIMEOUT_SECONDS` | 300 | `profile_all.sh` 超时上限（秒） |
| `HELLO_GPU_ARCH` | 未设置（由 rocminfo 检测） | 仅接受 `gfx1100`、`gfx1151` 或 `gfx1201` 作为显式编译 target；不代表硬件已验证 |

### 边界测试尺寸

- `N=1`：单元素
- `N=31`：小于 wave size (32)
- `N=32`：刚好一个 wave
- `N=33`：超过一个 wave
- `N=255, 256, 257`：block 边界
- `N=1027`：不是 4 的倍数，用于测试 float4 尾部处理

---


## 本章目标、前置知识与产物

> 如果把 GPU 算子比作一组流水线任务，Element-Wise（逐元素）就是最容易看清的一类：输出位置 `i` 只需要输入位置 `i`，不必等待其他位置。我们从 Vector Add（向量加法）开始，用同一道题走两条路线。HIP 篇把线程、地址与显存访问拆开讲；Triton 篇用一个 program 处理一块数据，先建立能快速修改的最小模板。

第 4 章已经带你第一次跑通 Vector Add，第 5–7 章已经介绍可信计时、kernel trace 与 Roofline 读图。本章不重走安装和完整 Roofline 推导，而是第一次回答四个“算子优化”问题：**同一个计算怎样用两种编程范式表达、数据怎样分给并行执行单元、地址怎样排列、改动是否真的有效。**

学完后，你需要能交付三样东西：一份覆盖尾部输入的正确性结果；一张可追溯到实验记录的 benchmark/profile 对照；一份写清负结果、适用边界和下一步的实验记录。

你不必把两条路线一次读完。先按自己的目标选择：

| 阅读路线 | 更适合谁 | 建议顺序 | 能带走什么 |
| ---- | ---- | ---- | ---- |
| 完整路线 | 第一次系统学习 GPU Kernel，希望理解两种写法 | 8.1 → 8.2 → 8.3 → 8.4 → 8.5 → 8.6 → 8.7 → 8.8 → 8.9 | 同一个算子从数学语义到实验闭环的全貌 |
| [HIP 路线](#_8-4-hip-从标量线程到受控访存实验) | 想看清 thread、wavefront、显存地址与向量加载，愿意写 C++ | 8.1 → 8.2 → 8.3 → 8.4 → 8.6 → 8.8 → 8.9 | 更强的底层控制与性能分析入口 |
| [Triton 路线](#_8-5-triton-从最小-tile-到参数实验) | 熟悉 Python/PyTorch，想先快速写出可验证的自定义算子 | 8.1 → 8.2 → 8.3 → 8.5 → 8.6 → 8.8 → 8.9 | program/tile/mask 的最小心智模型 |

> **提示**

HIP 常从“当前 thread 得到哪个标量下标”出发；Triton 常从“当前 program 生成哪一块下标”出发。两者最后都必须回答同一件事：**读了哪些地址，算了什么，写回哪里。**


## 8.1 先认识 Element-Wise


### 8.1.1 不看名字，先看依赖关系

判断一个算子是不是逐元素，最稳妥的方法不是背算子表，而是观察一个输出依赖哪些输入。

Vector Add 的定义是：

$$
C_i = A_i + B_i, \qquad 0 \le i \lt N
$$

想得到 `C[3]`，只需要 `A[3]` 与 `B[3]`。`A[0]`、`A[1]` 或其他位置都不会影响它。因此不同输出位置之间没有数据依赖，可以彼此独立地计算。


Vector Add 的逐元素依赖：当前位置只读取对应的两个输入位置。

如 上图 所示，动画每一步只激活同一列中的 `A[i]`、`B[i]` 与 `C[i]`。真实 GPU 会并行处理许多列；这里故意逐步播放，是为了让依赖关系更容易看清。

同一模式还包括：

| 算子 | 单个输出的表达式 | `Y[i]` 依赖什么 |
| ---- | ---- | ---- |
| Add | `Y[i] = A[i] + B[i]` | `A[i]`、`B[i]` |
| ReLU | `Y[i] = max(X[i], 0)` | `X[i]` |
| Scale & Bias | `Y[i] = alpha * X[i] + beta` | `X[i]` 与两个标量 |
| Clamp | `Y[i] = min(max(X[i], low), high)` | `X[i]` 与上下界 |

它们的计算表达式不同，但数据划分方法相似：先把互不依赖的下标分出去，再处理对应元素。HIP 与 Triton 对“怎样分出去”给出了不同的源码视角。


### 8.1.2 同一个 Vector Add，两种编程范式

上一节回答了“算什么”：每个位置独立完成一次加法。现在回答“怎样描述谁来算”。这两件事不要混在一起：**Element-Wise 是算子的数据依赖类别，HIP 与 Triton 是表达这类计算的两种编程范式。**

Triton 官方编程指南用下面两句话对照两种视角：

- CUDA 式线程级写法：`Scalar Program, Blocked Threads`；本章的 HIP 写法采用相同的 thread-centric SIMT 视角。
- Triton 分块写法：`Blocked Program, Scalar Threads`。

先不用背英文。对初学者更有用的翻译是：**HIP kernel 正文站在一个 thread 的位置，Triton kernel 正文站在一个 program 的位置。**


同一个 Vector Add 的两种分工视角：HIP 从标量线程与标量下标出发，Triton 从 program 与一块逻辑下标出发；两边最终只写回有效位置。

如 上图 所示，图中故意使用 `N=6`，但让启动范围覆盖位置 `0–7`。这样既能看到两种划分方式，也能看到它们怎样关闭最后两个越界位置。

先把真实 API 收起来，只看两段结构：

```text
HIP
当前 thread → 一个标量下标 i
如果 i 有效：C[i] = A[i] + B[i]

Triton
当前 program → 一组逻辑下标 offsets
对有效位置：C[offsets] = A[offsets] + B[offsets]
```

HIP 会启动许多 thread，每个 thread 概念上执行同一份 kernel 正文，只是得到的标量下标不同。“一个 thread 只算一个元素”只是本章 `HIP v0` 的起点；后面的版本可以让一个 thread 处理多个元素。

Triton 则让一个 program 直接描述一组逻辑下标上的计算。后文的 `tl.arange` 用来生成这组下标，它**不会创建同样数量的线程**；逻辑位置怎样映射到底层线程，由编译器继续完成。

把两段结构按同一组问题对齐：

| 问题 | HIP v0：线程级视角 | Triton t0：分块视角 |
| ---- | ---- | ---- |
| kernel 正文直接描述谁 | 一个 thread | 一个 program instance |
| 下标是什么形状 | 标量 `index` | 一块 `offsets` |
| 本例每个实例处理什么 | 一个输出位置 | 一个输出 tile |
| 怎样保护尾部 | 每个 thread 执行标量 `if` | mask 逐位置保护整块 load/store |
| 源码显式控制的组织层次 | thread 与 block | program 与 tile；底层线程布局交给编译器 |

> **注意**

HIP 的 `blockDim.x` 是一个 block 中显式启动的 thread 数；Triton 的 `BLOCK_SIZE` 是一个 program 逻辑覆盖的元素数。即使两者都写成 `256`，含义也不同。Triton program 不等于 HIP thread，tile 中的逻辑位置也不与某个硬件线程固定一一对应。

后面读代码时，可以反复用这组顺序翻译：

```text
HIP：    当前 thread → 标量 index   → 标量 load / add / store
Triton： 当前 program → 一块 offsets → 分块 load / add / store
```

两条路线改变的是源码直接控制的层次，不是 Vector Add 的数学语义。它们都要覆盖相同的有效下标，也都要经过正确性检查与实测，不能因为抽象层次更高或更低就预设谁更快。真实的 HIP API 留到 [8.4](#_8-4-hip-从标量线程到受控访存实验)，Triton 最小模板留到 [8.5](#_8-5-triton-从最小-tile-到参数实验) 再逐行展开。


### 8.1.3 什么不属于这一类

如果输出 `Y[i]` 需要一整段输入，事情就变了。例如 Sum Reduction 要把很多输入合成一个值，线程之间必须协作；Softmax 还要先求一行最大值与总和。它们分别是[第 9 章 Reduction](../../docs/part2-kernels/chapter9/index.md)和[第 10 章 Normalization](../../docs/part2-kernels/chapter10/index.md)的主角。

因此，“输入输出形状相同”不是逐元素的充分条件。真正重要的是**输出位置之间能不能独立完成**。


## 8.2 固定数学语义与正确性标准


### 8.2.1 先用 4 个元素手算

给定：

```text
A = [2, -1, 4, 3]
B = [7,  3, -1, 2]
```

逐位置相加：

```text
C[0] = 2  + 7  = 9
C[1] = -1 + 3  = 2
C[2] = 4  + -1 = 3
C[3] = 3  + 2  = 5

C = [9, 2, 3, 5]
```

因此，“输入输出形状相同”不是逐元素的充分条件。关键在于**每个输出位置是否能独立完成计算**。


### 8.2.2 固定同一份裁判规则

本章配套程序让 HIP 与 Triton 使用相同规则：

| 项目 | 固定方式 |
| ---- | ---- |
| 数学语义 | `C[i] = A[i] + B[i]` |
| 数据类型 | 32 位浮点数（FP32） |
| 输入 | 用同一确定性整数公式生成，再转成 FP32 |
| 参考结果 | CPU 逐元素执行同一个 FP32 加法 |
| 边界 | 覆盖小于 wave、刚好等于 block、比 block 多 1、不是 4 的倍数等长度 |
| 正确性 | 正式计时前检查一次，计时后再检查一次 |
| 计时范围 | 只计 GPU kernel，不含分配、输入生成与 Host-to-Device 拷贝 |

为什么需要检查奇怪的长度？因为真实输入不会永远刚好等于 block size 的整数倍。若 `N=1027`，最后一个 block 只有少数位置有效；没有边界保护的 kernel 会访问数组外部。

PyTorch 参考写法只有一行：

```python
reference = input_a + input_b
```

短不代表可以省略它。自定义 kernel 的第一个目标永远是与参考结果一致；一个错误但很快的 kernel 没有比较价值。


### 8.2.3 把“尾部”加入裁判规则

边界长度专门检查最后一组不完整的位置。例如 `N=13`、候选下标为 `8–15` 时，只有 `8–12` 有效：HIP 由各 thread 的标量 `if` 关闭越界下标，Triton 由 mask 逐位置关闭越界 load/store。这是在落实 上图 的同一条规则，不再引入第三种分工方式。


## 8.3 建立成本模型和瓶颈假设


### 8.3.1 真正忙的可能不是加法器

对一个 FP32 输出元素，最理想的逻辑工作量是：

| 动作 | 数量 | 逻辑字节 |
| ---- | ----: | ----: |
| 读取 `A[i]` | 1 个 FP32 | 4 Byte |
| 读取 `B[i]` | 1 个 FP32 | 4 Byte |
| 执行加法 | 1 次浮点加法 | 1 FLOP |
| 写回 `C[i]` | 1 个 FP32 | 4 Byte |
| 合计 |  | 12 Byte + 1 FLOP |

所以它的理想算术强度是：

$$
AI_{logical} = \frac{1\ \text{FLOP}}{12\ \text{Byte}}
             \approx 0.0833\ \text{FLOP/Byte}
$$

这只是从算子语义推导出的**逻辑下界**，不是性能实测。缓存命中、未合并访问、对齐和实际内存事务都可能让物理流量不同。

从这个比例可以提出一个等待实验检验的假设：

> Vector Add 每搬 12 Byte 只做 1 次加法，当前大 shape 可能更容易受显存带宽限制，而不是受浮点计算吞吐限制。

第 7 章已经解释怎样在 Roofline 上读工作点；这里不再重推硬件参考线，只保留与当前算子直接相关的假设：Vector Add 的逻辑算术强度很低，当前大 shape 更可能先受数据搬运限制。注意用词仍是“更可能”。后文会用 Radeon RX 9070 XT 上的 GPU event 时间和受控地址实验检验它。即使逻辑有效带宽较高，也只能说明结果与带宽受限假设一致；没有物理流量计数器时，请把逻辑字节当作算法口径，不直接解释为显存事务。


### 8.3.2 有效带宽怎样算

本章统一用计时区间中的逻辑字节计算有效带宽：

$$
BW_{effective} = \frac{3 \times N \times 4\ \text{Byte}}{t}
$$

其中 `t` 是一次 kernel 的 GPU event 时间。这个指标适合在**相同语义、相同 shape、相同计时范围**下比较版本，但它不等于内存控制器实际传输了多少字节。物理流量必须由可用的硬件计数器或更进一步的分析支持。


## 8.4 HIP：从标量线程到受控访存实验


### 8.4.1 这条路线适合谁

HIP 篇更适合下面的读者：

- 想知道 block、thread、wavefront 最后怎样变成具体地址；
- 愿意读少量 C++，并希望控制 grid、加载类型与尾部路径；
- 后续想继续学习 LDS、Wave Shuffle、VGPR 等更底层机制；
- 遇到性能问题时，希望能从源码一路追到 kernel trace。

第一次读 HIP 不需要先记住所有硬件名词。本节只反复使用四个对象：

```text
grid 里有多个 block
block 里有多个 thread
相邻 thread 以 wavefront 为硬件执行组
thread 最终访问全局内存中的地址
```

用一组具体数字锚定：本章 `N=16,777,216`、`blockDim.x=256`，那么 grid 需要 `65,536` 个 block；每个 block 里的 256 个 thread 按 32 个一组编成 8 个 wavefront；每个 thread 用 `blockIdx.x × 256 + threadIdx.x` 算出自己负责的那一个地址。后文所有版本只是在改变“哪个 thread 访问哪个地址、一次访问多少字节”。


### 8.4.2 HIP v0：一个线程处理一个元素

最短的正确 kernel 是：

```cpp
__global__ void vector_add_v0(const float* input_a,
                              const float* input_b,
                              float* output,
                              std::size_t size) {
    const std::size_t index =
        static_cast<std::size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (index < size) {
        output[index] = input_a[index] + input_b[index];
    }
}
```

先把索引公式拆开：

```text
blockIdx.x * blockDim.x    当前 block 之前有多少线程
+ threadIdx.x              当前线程在 block 内的位置
= index                    当前线程负责的全局元素下标
```

例如 `blockDim.x=256`，第 2 个 block 中的第 3 个线程得到：

```text
index = 2 × 256 + 3 = 515
```

它只处理 `C[515] = A[515] + B[515]`。最后一个 block 可能越过 `size`，所以 `if (index < size)` 不能省略。

当相邻线程得到相邻 `index` 时，它们也会访问相邻的 FP32 地址。AMD 的 HIP 性能指南把这种排列称为 coalesced memory access（合并访存）：硬件有机会把多个线程的请求组合成更少的内存事务。这里先把它当作**地址结构事实**；实际事务数与性能仍要测量。


### 8.4.3 HIP v1：怎样公平比较连续与跨步

一个常见教学错误是：连续版每线程处理 1 个元素，跨步版每线程处理 32 个元素，然后直接比较时间。此时改变的不只是地址顺序，还包括 grid、循环次数和每线程工作量，无法知道差异来自哪里。

本章改用一个受控实验。真实代码与动画缩略的比例如下，两边只改“同一轮内 lane 到地址的排列”，其余全部固定：

| 配置 | 真实 HIP 代码 | 上图 动画 |
| ---- | ---- | ---- |
| 执行单元 | 1 个 Wave32（32 个 lane） | 8 个 lane |
| 每 lane 处理 | 32 个元素 | 4 个元素 |
| 一轮覆盖 | 32 × 32 = 1024 元素 | 8 × 4 = 32 元素 |
| grid / block / 循环次数 / 逻辑字节 | 两边相同 | 两边相同 |


合并访存受控对照的缩略动画：两边处理同一批元素，只改变每一轮的下标排列。图中 32B 地址组是帮助观察聚集度的教学分桶，不是实测硬件事务计数；连续顺序每轮触及 1 组，跨步顺序每轮触及 4 组。

左侧连续版在第 `round` 轮使用：

```text
index = base + round × wave_size + lane
```

右侧跨步版只交换两个维度：

```text
index = base + lane × rounds + round
```

四轮动画结束后，两边都覆盖下标 `0–31`，没有重复也没有遗漏。连续顺序的累计组访问依次为 `1 / 2 / 3 / 4`，跨步顺序依次为 `4 / 8 / 12 / 16`。这仍是地址分组层面的教学计数；正式 HIP 代码保持相同的 grid、block、每 lane 循环次数、加法次数和逻辑字节，物理事务与性能由后续实测判断。

```cpp
// hip-v1-contiguous
index = base + round * 32 + lane;

// hip-v1-strided
index = base + lane * 32 + round;
```

这个实验要验证的不是“跨步一定慢多少”，而是：

> 在当前 9070XT、当前 shape 与当前编译结果下，只改变 wave 内地址顺序，kernel 时间和 trace 是否出现可重复差异？

2026-07-19 最终 curated evidence 给出了可重复差异：连续版的三进程 median 是 `0.369584 ms`，跨步版是 `2.567170 ms`，跨步版用时约为连续版的 `6.95×`。两者的 kernel trace 都记录到相同的 `524,288` 个 work-item、workgroup size 256、VGPR 16、SGPR 128、LDS 0 Byte 和 scratch 0 Byte；grid、循环次数、算术与逻辑字节也相同。

因此，当前证据支持“Wave32 同轮地址顺序显著影响这个 Vector Add”的判断。它仍没有直接数出物理显存事务，所以更严格的措辞是：**受控地址变化与约 `6.95×` 时间差同时出现，并且现有 trace 资源字段没有提供其他差异。**


### 8.4.4 HIP v2：Grid-Stride Loop 让线程重复工作

v0 启动足够多的线程，让每个线程只处理一个位置。另一种常见写法是限制 grid，让线程每隔整个 grid 的跨度继续处理下一个元素：

```cpp
const std::size_t thread =
    static_cast<std::size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
const std::size_t grid_stride =
    static_cast<std::size_t>(gridDim.x) * blockDim.x;

for (std::size_t index = thread;
     index < size;
     index += grid_stride) {
    output[index] = input_a[index] + input_b[index];
}
```

假设 grid 一共包含 1024 个线程：

```text
线程 0：0, 1024, 2048, ...
线程 1：1, 1025, 2049, ...
线程 2：2, 1026, 2050, ...
```

在同一轮循环中，相邻线程依旧访问相邻元素，因此 Grid-Stride 与“连续访存”并不冲突。

本章代码默认把 v2 的 grid 上限设为“设备 CU 数量 × 8”，并把最终 grid 写入 `RESULT`。这只是待验证的起点，不是通用最优值。减少 block 数可能降低调度开销，也可能让并行度不足；两种方向都要由实测回答。


### 8.4.5 HIP v3：`float4` 与尾部不是一回事

FP32 标量占 4 Byte，`float4` 把 4 个 FP32 组合成 16 Byte 类型。v3 先把完整的四元素组交给向量路径，再用标量处理最后 `0–3` 个元素。


`N=18` 时的 `float4` 源码分组与标量尾部。动画只画输入 A 的读取；输入 B 的读取与输出 C 的写回采用同样分组。该分组不等同于物理显存事务一定减少。

核心结构是：

```cpp
const std::size_t vector_count = size / 4;

for (std::size_t v = thread; v < vector_count; v += grid_stride) {
    const float4 a = input_a4[v];
    const float4 b = input_b4[v];
    output4[v] = make_float4(
        a.x + b.x, a.y + b.y, a.z + b.z, a.w + b.w);
}

for (std::size_t i = vector_count * 4 + thread;
     i < size;
     i += grid_stride) {
    output[i] = input_a[i] + input_b[i];
}
```

这里有三个容易混淆的事实：

1. `hipMalloc` 返回的基地址满足本例向量类型的对齐要求，但从任意偏移地址强转成 `float4*` 未必安全。
2. 使用 `float4` 只说明源码请求了向量类型，不自动证明最终指令数量或显存事务减少。
3. `N % 4 != 0` 时，尾部路径是正确性要求，不是可选优化。

因此 v3 仍然要检查编译结果、边界输入、kernel trace 和时间。若它没有变快，这也是有效结果：说明“源码向量化必然提速”的假设在当前条件下没有成立。


### 8.4.6 HIP 路线当前能下什么结论

HIP ladder 已经把三个问题拆开：v1 只控制同一轮的地址顺序，v2 改变 grid 与每线程工作方式，v3 再引入源码向量类型和尾部路径。单看代码无法给它们排快慢；统一的正确性、benchmark 和 trace 数据放在 [8.6](#_8-6-正确性、benchmark-与-profiling)，负结果与边界在 [8.8](#_8-8-负结果、适用边界与下一步) 汇总。

完整实现位于 `code/part2-kernels/chapter8/vector_add_hip.hip`。


## 8.5 Triton：从最小 Tile 到参数实验


### 8.5.1 这条路线适合谁

Triton 篇更适合下面的读者：

- 已经会写 Python 或 PyTorch，希望先缩短“想法到可运行 kernel”的距离；
- 想用一组 offsets 表达一块数据，不急着手工管理每个 thread；
- 希望快速修改算子表达式、block size 和 program 网格；
- 后续想把 Triton 最小模板迁移到新的逐元素练习。

Triton 不是“无需理解硬件”。它把许多线程级细节交给编译器，但 program 怎样切 tile、地址是否连续、mask 是否正确、参数是否合适，仍由程序员负责。


### 8.5.2 最小 kernel：七行建立完整数据路径

```python
@triton.jit
def vector_add_kernel(a_ptr, b_ptr, c_ptr, size,
                      BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    valid = offsets < size
    a = tl.load(a_ptr + offsets, mask=valid, other=0.0)
    b = tl.load(b_ptr + offsets, mask=valid, other=0.0)
    tl.store(c_ptr + offsets, a + b, mask=valid)
```

逐行看，不需要先背语法：

1. `@triton.jit` 告诉 Triton：下面定义的是要即时编译的 kernel。
2. `tl.program_id(0)` 取得当前 program 在一维网格中的编号。
3. `tl.arange(0, BLOCK_SIZE)` 生成一个 tile 内的局部下标。
4. `pid * BLOCK_SIZE` 把局部下标移动到当前 tile 的起点。
5. `valid` 标记哪些下标没有越过 `size`。
6. 两次 `tl.load` 读取相同 offsets 的 A/B。
7. `tl.store` 把逐元素加法写回相同 offsets 的 C。

最值得注意的是：`offsets` 不是一个 Python 整数，而是一整组下标。Triton 代码看起来像在操作向量，编译器再把这块工作映射到底层 GPU 执行资源。


### 8.5.3 Host wrapper 负责准备输出与网格

kernel 之外还需要 Host 侧代码：

```python
def launch(input_a, input_b, output, block_size):
    size = output.numel()
    grid = (triton.cdiv(size, block_size),)
    vector_add_kernel[grid](
        input_a,
        input_b,
        output,
        size,
        BLOCK_SIZE=block_size,
        num_warps=4,
    )
```

`triton.cdiv(size, block_size)` 是向上取整。`N=13`、`BLOCK_SIZE=8` 时需要 2 个 program；第二个 program 生成 `8–15`，再由 mask 关闭 `13–15`。

配套脚本使用 `torch.cuda.Event` 与 `device="cuda"`。在 ROCm 版 PyTorch 中看到 `cuda` 不表示代码跑到了 NVIDIA GPU；PyTorch 官方为了兼容现有生态，HIP 后端继续复用 `torch.cuda` 接口，可通过 `torch.version.hip` 确认当前构建。


### 8.5.4 Triton t0/t1：先只改一个参数

本章保留同一个 kernel，只改变 `BLOCK_SIZE`：

| 版本 | `BLOCK_SIZE` | `num_warps` | 改变了什么 |
| ---- | ----: | ----: | ---- |
| `triton-t0` | 256 | 4 | 最小正确 baseline |
| `triton-t1` | 1024 | 4 | 每个 program 覆盖更多元素，program 数量减少 |

这里把 `num_warps` 固定为 4，是为了让 block size 成为主要变量。`num_warps` 是 Triton 的元参数，指定编译一个 program 时使用多少个 warp/wave 执行组；它大致对应 HIP 语境里“一个 block 占几个 wavefront”，但 tile 元素怎样落到 lane 与寄存器仍由编译器决定。`BLOCK_SIZE` 更大可能减少 program 数量，也可能改变资源使用与调度；实测前不要把 t1 称为“优化版”。


### 8.5.5 用 Triton-viz 把 offsets 和 mask 展开

只看七行 kernel，初学者最容易卡在两个问题：`offsets` 到底有哪些数？最后一个 program 的越界位置发生了什么？


`N=13, BLOCK_SIZE=8` 的 Triton program、offsets、逐元素加法与尾部 mask：同一组逻辑 tile 位置依次读取 A、读取 B、执行 `A+B` 并写回 C，`mask=false` 的位置不会产生越界访问。

上图 是为正文重画的步骤动画。仓库还提供真实的 Triton-viz trace 脚本：

```python
@triton_viz.trace("tracer")
def traced_vector_add_kernel(...):
    # 与正文相同的 offsets / mask / load / store
    ...
```

待安装本篇环境后，可以启动实时可视化：

```bash
cd code/part2-kernels
python chapter8/visualize_triton.py --size 13 --block 8 --launch
```

浏览器打开 `http://127.0.0.1:5001` 后，Triton-viz 会通过 CPU interpreter 展开 load/store 地址，基础可视化不要求 GPU。它适合回答“访问了哪里、mask 是否挡住越界”，**不用于测量 9070XT 性能**。性能仍由原生 Ubuntu 实验机上的 GPU event 与 `rocprofv3` 负责。

可视化脚本使用当前平台已经安装的依赖版本；缺少 Triton-viz 时记录 SKIP，并打印可选的人工安装提示。本章只给出仓库中已经维护的实时启动入口，不额外提供未进入当前复跑契约的 `.tvz` 保存命令。

![Triton-viz 实时界面，左侧显示 program 控件和源码位置，右侧显示有效与被 mask 的元素](../../docs/part2-kernels/chapter8/images/triton-viz-vector-add.png)

本章配套的 Triton-viz 实时界面：拖动 program 滑块切换 program，右侧青色位置表示有效元素，灰色位置表示尾部 mask。

上图 左侧箭头定位到当前 `tl.load`，右侧青色位置表示有效元素，灰色位置表示尾部 mask。这张界面图只用于辅助读取地址与 mask，不是 9070XT 性能证据。


### 8.5.6 Triton 路线当前结果

Triton ladder 只把 `BLOCK_SIZE` 从 256 改为 1024，并保持 `num_warps=4`。更大的 tile 同时减少 program 数并改变资源需求，所以仍要把时间范围和 trace 放在一起读。统一结果见 [8.6](#_8-6-正确性、benchmark-与-profiling)，不要只看到 program 数减少就提前宣布胜负。

完整实现位于 `code/part2-kernels/chapter8/vector_add_triton.py`；可视化入口位于 `code/part2-kernels/chapter8/visualize_triton.py`。


## 8.6 正确性、Benchmark 与 Profiling

本节按 docs 的正式顺序执行：环境与依赖检查 → 边界正确性 → 3 次独立 benchmark → 逐实现 profiling → evidence 汇总。`RUN_CH8_FORMAL=True` 为默认值，因此从头 Run All 会直接尝试完整流程。

为保护仓库中已提交的 gfx1201 reference，Notebook 不直接在 `code/part2-kernels/chapter8/` 下运行会改写 evidence 的脚本。它会把现有 runner 与 `common/` 复制到临时目录执行，再把本次 `logs/`、`profiles/`、`evidence/` 保存到：

```text
runs/chapter8/<arch>/<UTC-run-id>/
```

当前平台结果与 `code/part2-kernels/chapter8/evidence/` 中的 gfx1201 reference 始终分开展示，不做跨 GPU 绝对性能排名。


### 8.6.1 环境、路径与依赖

云平台使用当前 Jupyter kernel 与平台 PATH 中已有的 ROCm 工具，不固定 Python 路径、wheel 源或软件版本。下面只检查能力是否存在，并原样打印当前版本。

依赖分为两类：

- 核心能力：`hipcc`、PyTorch ROCm、Triton；缺失时不能执行对应路径。
- 可选能力：Triton-viz；缺失时记录 SKIP，不影响正确性、benchmark 或 profiling。

缺少 Python 第三方库时，Notebook 只打印基于当前 `sys.executable` 的现场安装命令，不会自动联网、安装或修改环境。能否下载取决于平台网络、权限以及是否存在与当前 Python/ROCm/GPU 兼容的 wheel。`rocprofv3` 和 `hipcc` 属于 ROCm 系统工具，不应当作普通 PyPI 包自动安装。


In [ ]:
import subprocess
import json
import os
import re
import sys
import shutil
import importlib.util
from datetime import datetime, timezone
from pathlib import Path

# Robust repo root detection: search cwd and parents.
def find_repo_root():
    current = Path.cwd().resolve()
    for _ in range(6):
        if (current / "code").is_dir() and (current / "notebooks").is_dir():
            return current
        if (current / ".git").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return Path.cwd()


repo_root = find_repo_root()
code_dir = repo_root / "code" / "part2-kernels" / "chapter8"

print(f"仓库根目录: {repo_root}")
print(f"代码目录: {code_dir}")
print("云端镜像使用预装 ROCm / PyTorch / hipcc；本 notebook 跳过本地 uv 与 activate-rocm.sh 段，只检查额外第三方能力。")
print("\n检查关键文件:")
print(f"  HIP 源码: {(code_dir / 'vector_add_hip.hip').exists()}")
print(f"  Triton 源码: {(code_dir / 'vector_add_triton.py').exists()}")
print(f"  运行脚本: {(code_dir / 'run_all.sh').exists()}")

# HELLO_GPU_ARCH 仅指定编译 target，不设置 HSA override，也不用于证明已安装的硬件。
SUPPORTED_COMPILE_TARGETS = {"gfx1100", "gfx1151", "gfx1201"}
ARCH_TIMEOUT_SECONDS = 10
arch_override = os.environ.get("HELLO_GPU_ARCH", "").strip()
if arch_override:
    if arch_override not in SUPPORTED_COMPILE_TARGETS:
        raise RuntimeError(
            "HELLO_GPU_ARCH must be one of "
            f"{sorted(SUPPORTED_COMPILE_TARGETS)}; got {arch_override!r}"
        )
    arch = arch_override
    arch_source = "HELLO_GPU_ARCH compile-target override"
    print("架构来源是显式编译 target 覆盖，只用于编译，不等同于硬件验证")
else:
    try:
        result = subprocess.run(
            ["rocminfo"],
            capture_output=True,
            text=True,
            timeout=ARCH_TIMEOUT_SECONDS,
            check=False,
        )
    except subprocess.TimeoutExpired as error:
        raise RuntimeError(
            f"rocminfo 在 {ARCH_TIMEOUT_SECONDS}s 内未完成；请先检查环境，或填写已经确认过的 HELLO_GPU_ARCH"
        ) from error
    except FileNotFoundError as error:
        raise RuntimeError(
            "rocminfo 不可用；请先检查环境，或填写已经确认过的 HELLO_GPU_ARCH 作为明确的编译 target"
        ) from error
    if result.returncode != 0:
        stderr = result.stderr.strip() or "<empty>"
        stdout = result.stdout.strip() or "<empty>"
        raise RuntimeError(
            f"rocminfo 失败：returncode={result.returncode}; stderr={stderr}; stdout={stdout}"
        )
    candidates = sorted(
        set(re.findall(r"(?m)^\s*Name:\s*(gfx[0-9]+)\s*$", result.stdout))
    )
    if len(candidates) != 1:
        raise RuntimeError(
            "rocminfo 中应当只出现一个 GPU Agent 的 gfx 名称；"
            f"实际找到：{candidates or '无'}"
        )
    arch = candidates[0]
    arch_source = "rocminfo"

print(f"\narch={arch} arch_source={arch_source}")
if arch != "gfx1201":
    print(
        f"当前平台 {arch} 不是 RDNA4/gfx1201 参考平台；"
        "只比较正确性和平台内趋势，不与参考性能数字直接比较"
    )
if arch_source == "rocminfo":
    print("rocminfo GPU Agent Name 已确认本次检测候选；仍请把结果写入实验记录")

print("\n额外第三方能力检查:")
dependency_modules = {
    "triton": ("triton", "triton"),
    "numpy": ("numpy", "numpy"),
    "matplotlib": ("matplotlib", "matplotlib"),
    "triton-viz": ("triton_viz", "triton-viz"),
}
dependency_status = {}
for package, (module_name, install_package) in dependency_modules.items():
    try:
        available = importlib.util.find_spec(module_name) is not None
    except (ImportError, ValueError):
        available = False
    dependency_status[package] = available
    if available:
        print(f"  {package}: available")
    else:
        print(
            f"  {package}: unavailable; manual install: "
            f"`{sys.executable} -m pip install {install_package}`"
        )
print("  安装提示只供人工选择；Notebook 不会自动执行下载。")
rocprof_available = shutil.which("rocprofv3") is not None
print(f"  rocprofv3 (optional system capability): {'available' if rocprof_available else 'optional-unavailable'}")


### 8.6.2 执行完整正确性、Benchmark 与 Profiling

默认参数与 docs 一致：边界输入 `1、31、32、33、255、256、257、1027`，主输入 `N=16,777,216`，warmup=10，repeat=50，3 次独立进程复跑。`run_all.sh` 成功后才启动 `profile_all.sh`；benchmark 限时 900 秒，profiling 限时 300 秒，并完整显示 stdout、stderr 与 returncode。

如果当前目录不是 Git checkout，需要在运行 Notebook 前显式提供合法且已核对的 `SOURCE_COMMIT`。该字段只用于 evidence 溯源，不决定 GPU 能否执行。

本单元不锁定第三方库来源或版本。若 `torch`、`triton` 等库缺失，输出会显示当前解释器对应的手工安装命令；若库已经存在但 profiler 报 LLVM/运行时冲突，则属于兼容性问题，不能伪装成“缺少依赖”，也不会通过安装 Triton-viz 修复。


In [ ]:
import csv
import shlex
import signal
import tempfile

RUN_CH8_FORMAL = True
FORMAL_TIMEOUT_SECONDS = 900
PROFILE_TIMEOUT_SECONDS = 300
RUNNER_PYTHON = Path(sys.executable)
EXPECTED_IMPLEMENTATIONS = {
    "hip-v0",
    "hip-v1-contiguous",
    "hip-v1-strided",
    "hip-v2",
    "hip-v3",
    "triton-t0",
    "triton-t1",
}


def bundled_rocprofv3_available():
    """True when the active Python environment bundles a ROCm SDK whose
    rocprofv3 matches the bundled runtime (e.g. pip-installed torch ROCm
    wheels ship _rocm_sdk_core).  Profiling Triton with that tool avoids
    mixing the bundled runtime with a mismatched system rocprofv3."""
    try:
        import importlib.util
        import pathlib
        spec = importlib.util.find_spec("rocm_sdk")
        if spec is None or not spec.origin:
            return False
        sdk_root = pathlib.Path(spec.origin).resolve().parent.parent / "_rocm_sdk_core"
        return (
            (sdk_root / "bin" / "rocprofv3").is_file()
            and (sdk_root / "lib").is_dir()
        )
    except Exception:
        return False


def inspect_hipcc():
    hipcc_path = shutil.which("hipcc")
    if hipcc_path is None:
        return None
    try:
        result = subprocess.run(
            [hipcc_path, "--version"],
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        print(f"hipcc 无法检查：{error}")
        return None
    combined = f"{result.stdout}\n{result.stderr}"
    match = re.search(r"HIP version:\s*([^\s]+)", combined)
    if result.returncode != 0 or match is None:
        print(f"hipcc 版本检查失败：returncode={result.returncode}")
        print(combined)
        return None
    return {"path": hipcc_path, "version": match.group(1)}


def inspect_runner_python(python_executable):
    probe = r"""
import importlib.util
import json

required = ("torch", "triton", "numpy")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print(json.dumps({"status": "MISSING_PACKAGES", "missing": missing}))
    raise SystemExit(0)

try:
    import numpy
    import torch
    import triton
except Exception as error:
    print(json.dumps({
        "status": "IMPORT_OR_ABI_ERROR",
        "error_type": type(error).__name__,
        "error": str(error),
    }))
    raise SystemExit(0)

print(json.dumps({
    "status": "OK",
    "numpy": numpy.__version__,
    "torch": torch.__version__,
    "triton": triton.__version__,
    "torch_hip": torch.version.hip,
    "gpu_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "triton_viz_available": importlib.util.find_spec("triton_viz") is not None,
}))
"""
    try:
        result = subprocess.run(
            [str(python_executable), "-c", probe],
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        print(f"当前 Python 能力检查无法启动：{python_executable}: {error}")
        return None
    if result.returncode != 0:
        print(f"当前 Python 能力检查异常退出：returncode={result.returncode}")
        if result.stdout:
            print(f"stdout:\n{result.stdout}")
        if result.stderr:
            print(f"stderr:\n{result.stderr}")
        return None
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        print("当前 Python 能力检查失败：没有 JSON 输出")
        return None
    try:
        stack = json.loads(lines[-1])
    except json.JSONDecodeError as error:
        print(f"当前 Python 能力检查失败：最后一行不是 JSON：{error}")
        print(result.stdout)
        return None
    stack["python"] = str(python_executable)
    return stack

def run_visible(command, *, cwd, env, timeout_seconds, label):
    print(f"[{label}] timeout={timeout_seconds}s command={command}")
    try:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            start_new_session=(os.name == "posix"),
        )
    except FileNotFoundError as error:
        print(f"[{label}] status=NOT_STARTED: {error}")
        return None

    try:
        stdout, stderr = process.communicate(timeout=timeout_seconds)
    except subprocess.TimeoutExpired:
        if os.name == "posix":
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        else:
            process.terminate()
        try:
            stdout, stderr = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == "posix":
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            else:
                process.kill()
            stdout, stderr = process.communicate()
        if stdout:
            print(f"[{label}] partial stdout:\n{stdout}")
        if stderr:
            print(f"[{label}] partial stderr:\n{stderr}")
        print(f"[{label}] status=TIMEOUT after {timeout_seconds}s")
        return None

    if stdout:
        print(f"[{label}] stdout:\n{stdout}")
    if stderr:
        print(f"[{label}] stderr:\n{stderr}")
    print(f"[{label}] returncode={process.returncode}")
    return subprocess.CompletedProcess(
        command, process.returncode, stdout, stderr
    )


def source_commit_for_formal_run():
    provided = os.environ.get("SOURCE_COMMIT", "").strip()
    if provided and re.fullmatch(r"[0-9a-f]{7,40}", provided) is None:
        print("SOURCE_COMMIT 格式无效；必须是 7–40 位小写十六进制 Git SHA")
        return None

    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        if provided:
            print(f"git HEAD 不可用，使用已显式提供的 SOURCE_COMMIT：{provided}（{error}）")
            return provided
        print(f"git HEAD 不可用且未提供 SOURCE_COMMIT：{error}")
        return None

    head = result.stdout.strip()
    if result.returncode == 0 and re.fullmatch(r"[0-9a-f]{40}", head):
        if provided and not head.startswith(provided):
            print(f"SOURCE_COMMIT={provided} 与当前 HEAD={head} 不匹配")
            return None
        verified = provided or head
        print(f"源码身份：{verified}")
        return verified

    if provided:
        print("git HEAD 不可用，使用已显式提供并人工核对的 SOURCE_COMMIT")
        return provided
    print(
        "无法确认源码身份；若云端副本不含 .git，请先人工核对后设置 "
        "SOURCE_COMMIT=<7-40位小写Git SHA>"
    )
    return None


def prepare_isolated_runner(temporary_root):
    temporary_part2 = temporary_root / "part2-kernels"
    temporary_chapter = temporary_part2 / "chapter8"
    ignore_generated = shutil.ignore_patterns(
        "evidence", "logs", "profiles", "runs", "__pycache__", "*.pyc"
    )
    shutil.copytree(code_dir, temporary_chapter, ignore=ignore_generated)
    shutil.copytree(
        code_dir.parent / "common",
        temporary_part2 / "common",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"),
    )
    return temporary_part2, temporary_chapter


def preserve_run_artifacts(temporary_chapter, destination, status):
    destination.mkdir(parents=True, exist_ok=False)
    for name in ("logs", "profiles", "evidence"):
        source = temporary_chapter / name
        if source.exists():
            shutil.copytree(source, destination / name)
    (destination / "notebook_status.json").write_text(
        json.dumps(status, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"本次运行产物已保存：{destination}")


def read_ch8_evidence(evidence_dir):
    with (evidence_dir / "manifest.json").open(encoding="utf-8") as handle:
        manifest = json.load(handle)
    with (evidence_dir / "summary.csv").open(newline="", encoding="utf-8") as handle:
        summary_rows = list(csv.DictReader(handle))
    with (evidence_dir / "profile_summary.csv").open(
        newline="", encoding="utf-8"
    ) as handle:
        profile_rows = list(csv.DictReader(handle))

    summary_implementations = {row["implementation"] for row in summary_rows}
    profile_implementations = {row["implementation"] for row in profile_rows}
    if summary_implementations != EXPECTED_IMPLEMENTATIONS:
        raise RuntimeError(
            "summary.csv 必须恰好包含七个预期实现；"
            f"实际={sorted(summary_implementations)}"
        )
    if profile_implementations != EXPECTED_IMPLEMENTATIONS:
        raise RuntimeError(
            "profile_summary.csv 必须恰好包含七个预期实现；"
            f"实际={sorted(profile_implementations)}"
        )
    invalid_rows = [
        row["implementation"]
        for row in summary_rows
        if row.get("correct") != "OK" or row.get("run_count") != "3"
    ]
    if invalid_rows:
        raise RuntimeError(
            "每个 summary 行必须 correct=OK 且 run_count=3；"
            f"异常实现={invalid_rows}"
        )
    return manifest, summary_rows, profile_rows


def display_ch8_evidence(evidence_dir, label):
    manifest, summary_rows, profile_rows = read_ch8_evidence(evidence_dir)
    benchmark = manifest.get("benchmark", {})
    print(f"\n[{label}] evidence: {evidence_dir}")
    print(
        "manifest: "
        f"gpu_arch={benchmark.get('gpu_arch', '<unknown>')}; "
        f"size={benchmark.get('size', '<unknown>')}; "
        f"warmup={benchmark.get('warmup', '<unknown>')}; "
        f"repeat={benchmark.get('repeat', '<unknown>')}"
    )
    print("summary.csv")
    for row in summary_rows:
        print(
            f"  {row['implementation']}: correct={row['correct']}, "
            f"runs={row['run_count']}, median_ms={row['median_ms']}, "
            f"bandwidth={row['effective_bandwidth_gbs']} GB/s"
        )
    print("profile_summary.csv")
    for row in profile_rows:
        print(
            f"  {row['implementation']}: "
            f"dispatches={row.get('trace_dispatches', 'unavailable')}, "
            f"grid={row.get('trace_grid_size_x', 'unavailable')}, "
            f"workgroup={row.get('trace_workgroup_size_x', 'unavailable')}, "
            f"vgpr={row.get('trace_vgpr_count', 'unavailable')}, "
            f"sgpr={row.get('trace_sgpr_count', 'unavailable')}"
        )
    return manifest, summary_rows, profile_rows



def read_env_values(path):
    values = {}
    if not path.is_file():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        key, separator, value = line.partition("=")
        if separator and key:
            values[key] = value
    return values


def refresh_partial_profile_evidence(chapter_dir, source_commit, env):
    """Refresh profile_summary.csv from any completed traces after profiler failure."""
    profile_dir = chapter_dir / "profiles"
    trace_paths = sorted(profile_dir.glob("*_kernel_trace.csv"))
    trace_labels = [
        path.name.removesuffix("_kernel_trace.csv") for path in trace_paths
    ]
    if not trace_paths:
        print("Partial profile summary: no kernel trace files were produced")
        return False, trace_labels

    benchmark = read_env_values(chapter_dir / "logs" / "benchmark_manifest.env")
    required = {
        "source_commit",
        "source_sha256",
        "size",
        "hip_block",
        "triton_t0_block",
        "triton_block",
        "seed",
        "gpu_arch",
    }
    missing = sorted(required - benchmark.keys())
    if missing:
        print(
            "Partial profile summary was not refreshed: "
            f"benchmark manifest missing {missing}"
        )
        return False, trace_labels

    profile_dir.mkdir(parents=True, exist_ok=True)
    profile_config = {
        "source_commit": benchmark["source_commit"],
        "source_sha256": benchmark["source_sha256"],
        "size": benchmark["size"],
        "hip_block": benchmark["hip_block"],
        "triton_t0_block": benchmark["triton_t0_block"],
        "triton_block": benchmark["triton_block"],
        "warmup": env.get("PROFILE_WARMUP", "0"),
        "repeat": env.get("PROFILE_REPEAT", "10"),
        "seed": benchmark["seed"],
        "gpu_arch": benchmark["gpu_arch"],
        "grid": benchmark.get("grid", "auto"),
    }
    (profile_dir / "profile_config.env").write_text(
        "".join(f"{key}={value}\n" for key, value in profile_config.items()),
        encoding="utf-8",
    )

    summary_result = run_visible(
        [
            str(RUNNER_PYTHON),
            str(chapter_dir / "summarize_results.py"),
            "--chapter-dir",
            str(chapter_dir),
            "--git-commit",
            source_commit,
        ],
        cwd=chapter_dir.parent,
        env=env,
        timeout_seconds=60,
        label="Chapter8 partial profile summary refresh",
    )
    refreshed = summary_result is not None and summary_result.returncode == 0
    if refreshed:
        print(
            "Partial profile summary refreshed from raw traces: "
            + ", ".join(trace_labels)
        )
        print(
            "Missing implementations remain unavailable; "
            "profile_complete stays False."
        )
    return refreshed, trace_labels


CH8_CORRECTNESS_COMPLETE = False
CH8_BENCHMARK_COMPLETE = False
CH8_PROFILE_COMPLETE = False
CH8_PARTIAL_PROFILE_SUMMARY = False
CH8_PROFILE_TRACE_LABELS = []
CH8_FORMAL_COMPLETE = False
CURRENT_RUN_DIR = None
CURRENT_EVIDENCE_DIR = None

if not RUN_CH8_FORMAL:
    print("完整流程由 RUN_CH8_FORMAL=False 显式关闭")
else:
    missing_core = []
    if shutil.which("bash") is None:
        missing_core.append("bash")
    hipcc_info = inspect_hipcc()
    if hipcc_info is None:
        missing_core.append("hipcc")

    runner_stack = inspect_runner_python(RUNNER_PYTHON)
    if hipcc_info is not None:
        print("Chapter8 HIP 编译器")
        print(f"- hipcc: {hipcc_info['path']}")
        print(f"- HIP version: {hipcc_info['version']}")
    if runner_stack is not None:
        print("Chapter8 当前 Python 能力")
        print(f"- python: {runner_stack['python']}")
        print(f"- status: {runner_stack['status']}")
        if runner_stack["status"] == "OK":
            print(f"- numpy: {runner_stack['numpy']}")
            print(f"- torch: {runner_stack['torch']}")
            print(f"- triton: {runner_stack['triton']}")
            print(f"- torch.version.hip: {runner_stack['torch_hip']}")
            print(f"- gpu_available: {runner_stack['gpu_available']}")
            print(f"- gpu_name: {runner_stack['gpu_name']}")
        elif runner_stack["status"] == "MISSING_PACKAGES":
            print(f"- missing: {runner_stack['missing']}")
        else:
            print(
                f"- import_error: {runner_stack.get('error_type')}: "
                f"{runner_stack.get('error')}"
            )

    runner_capabilities_ok = (
        runner_stack is not None and runner_stack.get("status") == "OK"
    )
    runner_gpu_is_available = (
        runner_capabilities_ok
        and runner_stack.get("gpu_available") is True
    )
    source_commit = (
        source_commit_for_formal_run()
        if not missing_core and runner_gpu_is_available
        else None
    )

    if missing_core:
        print(f"Chapter8 完整流程未启动：缺少系统核心能力 {missing_core}")
    elif runner_stack is None:
        print("Chapter8 完整流程未启动：当前 Python 能力探针本身失败。")
        print("这不是已确认的库缺失；请先查看上方 stdout/stderr。")
    elif runner_stack["status"] == "MISSING_PACKAGES":
        missing_packages = runner_stack["missing"]
        print(f"Chapter8 完整流程未启动：缺少第三方库 {missing_packages}")
        print(
            f"可人工尝试：{RUNNER_PYTHON} -m pip install "
            + " ".join(missing_packages)
        )
        print("能否现场下载取决于平台网络、权限和兼容 wheel。")
    elif runner_stack["status"] == "IMPORT_OR_ABI_ERROR":
        print("Chapter8 完整流程未启动：库已发现，但导入或 ABI 不兼容。")
        print(
            f"{runner_stack.get('error_type')}: {runner_stack.get('error')}"
        )
        print("不要把该状态当成缺包，也不要盲目重复 pip install。")
    elif not runner_gpu_is_available:
        print("Chapter8 完整流程未启动：当前 Python 看不到 ROCm GPU。")
        print(
            "请检查 /dev/kfd、/dev/dri 权限、容器 GPU 映射及 PyTorch ROCm 构建；"
            "这不等同于缺少普通 Python 包。"
        )
    elif source_commit is None:
        print("Chapter8 完整流程未启动：无法确认 SOURCE_COMMIT")
    else:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        CURRENT_RUN_DIR = repo_root / "runs" / "chapter8" / arch / run_id
        triton_viz_enabled = runner_stack["triton_viz_available"]
        if triton_viz_enabled:
            print("Triton-viz: available，将执行 visualization 检查")
        else:
            print("Triton-viz: SKIP（依赖缺失）；继续正确性、benchmark 与 profiling")

        formal_env = os.environ.copy()
        formal_env.update({
            "GPU_ARCH": arch,
            "HELLO_GPU_ARCH": arch,
            "HELLO_GPU_SKIP_ACTIVATE": "1",
            "SOURCE_COMMIT": source_commit,
            "RUN_EDGE_CASES": "1",
            "RUN_TRITON_VIZ": "1" if triton_viz_enabled else "0",
            "INDEPENDENT_RUNS": "3",
            "SIZE": "16777216",
            "HIP_BLOCK": "256",
            "TRITON_BLOCK": "1024",
            "WARMUP": "10",
            "REPEAT": "50",
            "SEED": "20260716",
            "PROFILE_WARMUP": "0",
            "PROFILE_REPEAT": "10",
            "TRITON_PROFILE_MODE": (
                os.environ.get("TRITON_PROFILE_MODE")
                or ("direct" if bundled_rocprofv3_available() else "skip")
            ),
        })

        with tempfile.TemporaryDirectory(
            prefix=f"hello-gpu-chapter8-{arch}-"
        ) as temporary:
            temporary_root = Path(temporary)
            runner_bin = temporary_root / "bin"
            runner_bin.mkdir()
            python_shim = runner_bin / "python"
            python_shim.write_text(
                "#!/usr/bin/env bash\n"
                f"exec {shlex.quote(str(RUNNER_PYTHON))} \"$@\"\n",
                encoding="utf-8",
            )
            python_shim.chmod(0o755)
            formal_env["PATH"] = (
                f"{runner_bin}{os.pathsep}{formal_env.get('PATH', '')}"
            )
            print(f"临时 runner Python wrapper: {python_shim} -> {RUNNER_PYTHON}")

            temporary_part2, temporary_chapter = prepare_isolated_runner(
                temporary_root
            )
            benchmark_result = run_visible(
                ["bash", str(temporary_chapter / "run_all.sh")],
                cwd=temporary_part2,
                env=formal_env,
                timeout_seconds=FORMAL_TIMEOUT_SECONDS,
                label="Chapter8 correctness + benchmark",
            )
            CH8_BENCHMARK_COMPLETE = (
                benchmark_result is not None and benchmark_result.returncode == 0
            )
            CH8_CORRECTNESS_COMPLETE = CH8_BENCHMARK_COMPLETE

            if CH8_BENCHMARK_COMPLETE:
                profile_result = run_visible(
                    ["bash", str(temporary_chapter / "profile_all.sh")],
                    cwd=temporary_part2,
                    env=formal_env,
                    timeout_seconds=PROFILE_TIMEOUT_SECONDS,
                    label="Chapter8 profiling",
                )
                CH8_PROFILE_COMPLETE = (
                    profile_result is not None and profile_result.returncode == 0
                )
                CH8_PROFILE_TRACE_LABELS = sorted(
                    path.name.removesuffix("_kernel_trace.csv")
                    for path in (temporary_chapter / "profiles").glob(
                        "*_kernel_trace.csv"
                    )
                )
                if not CH8_PROFILE_COMPLETE:
                    (
                        CH8_PARTIAL_PROFILE_SUMMARY,
                        CH8_PROFILE_TRACE_LABELS,
                    ) = refresh_partial_profile_evidence(
                        temporary_chapter,
                        source_commit,
                        formal_env,
                    )
            else:
                print("Chapter8 profiling: NOT_STARTED（benchmark 未成功）")

            CH8_FORMAL_COMPLETE = (
                CH8_CORRECTNESS_COMPLETE
                and CH8_BENCHMARK_COMPLETE
                and CH8_PROFILE_COMPLETE
            )
            if not triton_viz_enabled:
                triton_viz_status = "SKIP"
            elif CH8_BENCHMARK_COMPLETE or (
                temporary_chapter / "logs" / "hip_benchmark.log"
            ).is_file():
                triton_viz_status = "PASS"
            else:
                triton_viz_status = "FAIL_OR_NOT_REACHED"

            run_status = {
                "arch": arch,
                "source_commit": source_commit,
                "runner_python": str(RUNNER_PYTHON),
                "hipcc": hipcc_info["path"],
                "hipcc_version": hipcc_info["version"],
                "numpy": runner_stack["numpy"],
                "torch": runner_stack["torch"],
                "triton": runner_stack["triton"],
                "torch_hip": runner_stack["torch_hip"],
                "triton_viz": triton_viz_status,
                "correctness_complete": CH8_CORRECTNESS_COMPLETE,
                "benchmark_complete": CH8_BENCHMARK_COMPLETE,
                "profile_complete": CH8_PROFILE_COMPLETE,
                "partial_profile_summary": CH8_PARTIAL_PROFILE_SUMMARY,
                "profile_trace_labels": CH8_PROFILE_TRACE_LABELS,
                "formal_complete": CH8_FORMAL_COMPLETE,
            }
            preserve_run_artifacts(
                temporary_chapter,
                CURRENT_RUN_DIR,
                run_status,
            )

        candidate_evidence = CURRENT_RUN_DIR / "evidence"
        required_evidence = (
            candidate_evidence / "manifest.json",
            candidate_evidence / "summary.csv",
            candidate_evidence / "profile_summary.csv",
        )
        if CH8_BENCHMARK_COMPLETE and all(
            path.is_file() for path in required_evidence
        ):
            try:
                read_ch8_evidence(candidate_evidence)
            except (OSError, KeyError, RuntimeError, csv.Error) as error:
                print(
                    "Current evidence is incomplete and will not be displayed: "
                    f"{error}"
                )
            else:
                CURRENT_EVIDENCE_DIR = candidate_evidence
        elif CH8_BENCHMARK_COMPLETE:
            missing_evidence = [
                str(path) for path in required_evidence if not path.is_file()
            ]
            print(
                "Current evidence is incomplete and will not be displayed; "
                f"missing={missing_evidence}"
            )

print("\nChapter8 执行状态")
print(f"- correctness_complete: {CH8_CORRECTNESS_COMPLETE}")
print(f"- benchmark_complete: {CH8_BENCHMARK_COMPLETE}")
print(f"- profile_complete: {CH8_PROFILE_COMPLETE}")
print(f"- partial_profile_summary: {CH8_PARTIAL_PROFILE_SUMMARY}")
print(f"- profile_trace_labels: {CH8_PROFILE_TRACE_LABELS}")
print(f"- formal_complete: {CH8_FORMAL_COMPLETE}")
if CURRENT_RUN_DIR is not None:
    print(f"- run_dir: {CURRENT_RUN_DIR}")


### 8.6.3 本次平台结果与 gfx1201 Reference

下方先读取本次隔离运行生成的 evidence（若 benchmark 成功），再读取仓库中已提交的 gfx1201 reference。两套结果分别打印，禁止把 reference 数字标成当前云端实测。


### 8.6.4 正确性矩阵（gfx1201 Reference）

以下表格是已提交的 gfx1201 reference；本次平台的七个实现与 `correct/run_count` 已由上方动态输出直接读取。

| Implementation | Runtime | Shape | Block | Grid | Correct | Max abs error |
| ---- | ---- | ----: | ----: | ----: | ---- | ----: |
| `hip-v0` | hip | 16777216 | 256 | 65536 | OK | 0.0 |
| `hip-v1-contiguous` | hip | 16777216 | 256 | 2048 | OK | 0.0 |
| `hip-v1-strided` | hip | 16777216 | 256 | 2048 | OK | 0.0 |
| `hip-v2` | hip | 16777216 | 256 | 256 | OK | 0.0 |
| `hip-v3` | hip | 16777216 | 256 | 256 | OK | 0.0 |
| `triton-t0` | triton | 16777216 | 256 | 65536 | OK | 0.0 |
| `triton-t1` | triton | 16777216 | 1024 | 16384 | OK | 0.0 |


### 8.6.5 Benchmark 口径与发布结果（gfx1201 Reference）

下表只展示已提交的 gfx1201 reference。每个进程先 warmup 10 次、正式计时 50 次；每个发布值先取进程内 median，再对 3 个独立进程的 median 取中位数。计时范围是 kernel-only GPU event，不含分配、输入生成和 Host-to-Device 拷贝。

| Implementation | Median (ms) | Run range (ms) | Logical effective bandwidth (GB/s) | Run range (GB/s) |
| ---- | ----: | ----: | ----: | ----: |
| `hip-v0` | 0.336324 | 0.336224–0.337965 | 598.609044 | 595.703362–598.787113 |
| `hip-v1-contiguous` | 0.369584 | 0.367805–0.370344 | 544.738374 | 543.620507–547.373904 |
| `hip-v1-strided` | 2.567170 | 2.539349–2.581509 | 78.423556 | 77.987935–79.282759 |
| `hip-v2` | 0.343484 | 0.341984–0.345644 | 586.130918 | 582.468070–588.701781 |
| `hip-v3` | 0.345224 | 0.344864–0.347444 | 583.176683 | 579.450457–583.785476 |
| `triton-t0` | 0.336204 | 0.335084–0.336803 | 598.823604 | 597.756836–600.824263 |
| `triton-t1` | 0.338504 | 0.338404–0.338844 | 594.753950 | 594.157167–594.929706 |

![七个 HIP 与 Triton Vector Add 实现的逻辑有效带宽；跨步 HIP 版本明显低于其余连续访问版本](../../docs/part2-kernels/chapter8/images/vector-add-ch7-bandwidth.png)

Radeon RX 9070 XT 上的 Vector Add 逻辑有效带宽；柱长为三进程中位数，误差线为三进程范围。

图和表中的带宽都按 `12 Byte × N / 时间` 计算，是方便同语义版本比较的**逻辑有效带宽**。它不等于内存控制器实际传输的物理 GDDR6 流量。



### 8.6.6 Profiling 字段（gfx1201 Reference）

`profile_all.sh` 为每个实现启动独立的 `rocprofv3` kernel trace。下表是只读的 gfx1201 已发布参考结果；dispatch 数、grid、workgroup 与资源字段不从源码反推。

| Implementation | Dispatches | Grid X | Workgroup X | LDS B | Scratch B | VGPR | Accum VGPR | SGPR |
| ---- | ----: | ----: | ----: | ----: | ----: | ----: | ----: | ----: |
| `hip-v0` | 6 | 16777216 | 256 | 0 | 0 | 8 | 0 | 128 |
| `hip-v1-contiguous` | 6 | 524288 | 256 | 0 | 0 | 16 | 0 | 128 |
| `hip-v1-strided` | 6 | 524288 | 256 | 0 | 0 | 16 | 0 | 128 |
| `hip-v2` | 6 | 65536 | 256 | 0 | 0 | 16 | 0 | 128 |
| `hip-v3` | 6 | 65536 | 256 | 0 | 0 | 16 | 0 | 128 |
| `triton-t0` | 6 | 8388608 | 128 | 0 | 0 | 8 | 0 | 128 |
| `triton-t1` | 6 | 2097152 | 128 | 0 | 0 | 24 | 0 | 128 |



### 8.6.7 有效带宽计算与解释

本次平台与 gfx1201 reference 都使用同一逻辑口径：

$$
BW_{effective} = \frac{3 \times N \times 4\ \text{Byte}}{t}
$$

其中 `t` 是 kernel-only GPU event 的进程内 median。该值用于同一平台、同一 shape、同一计时范围下比较实现，不等于物理 GDDR6 流量，也不用于跨 GPU 排名。


### 8.6.8 执行状态与证据对照

`CH8_FORMAL_COMPLETE=True` 仅在边界正确性、3 次独立 benchmark 和七个实现的 profiling 全部完成时成立。Triton profiling 要求 rocprofv3 与进程内 ROCm 运行时版本匹配：某些环境（如 pip 安装的 torch ROCm 轮子）自带 `_rocm_sdk_core`，与系统 rocprofv3 混用会触发符号缺失或工具重复注册；这类环境会自动选择 `TRITON_PROFILE_MODE=direct`，`profile_all.sh` 会改用环境自带的 rocprofv3 采集 Triton traces。未检测到自带配套工具时默认 `skip`（只采集可靠的 HIP traces，把 Triton profile 标为 `unavailable`），可通过环境变量显式覆盖。partial `profile_summary.csv` 会保留成功实现的真实字段；此时 `CH8_PARTIAL_PROFILE_SUMMARY=True`，但 `CH8_PROFILE_COMPLETE` 与 formal 状态仍为 False。Triton kernel 的正确性和 benchmark 仍照常执行；Triton-viz 缺失只显示 SKIP。


In [ ]:
REFERENCE_EVIDENCE_DIR = code_dir / "evidence"
reference_manifest, reference_summary_rows, reference_profile_rows = (
    display_ch8_evidence(REFERENCE_EVIDENCE_DIR, "gfx1201 reference")
)

if CURRENT_EVIDENCE_DIR is not None:
    current_manifest, current_summary_rows, current_profile_rows = (
        display_ch8_evidence(CURRENT_EVIDENCE_DIR, f"current platform {arch}")
    )
else:
    current_manifest = None
    current_summary_rows = []
    current_profile_rows = []
    print("\n[current platform] 本次 benchmark 未产生可发布 evidence；请查看上方 returncode 与 run_dir。")

if CH8_FORMAL_COMPLETE:
    print("\nChapter8 formal status: PASS")
elif CH8_BENCHMARK_COMPLETE:
    print("\nChapter8 formal status: PARTIAL（benchmark 完成，profiling 未完成）")
else:
    print("\nChapter8 formal status: FAIL（正确性/benchmark 未完成）")
print("注意：非 gfx1201 结果只用于当前平台内比较，不与 reference 做绝对性能排名。")


## 8.7 HIP 与 Triton 对照


### 8.7.1 把两种语言放回同一张数据流

| 要回答的问题 | HIP 写法 | Triton 写法 |
| ---- | ---- | ---- |
| 源码中显式编号的并行实例 | `blockIdx.x`、`threadIdx.x` 选出当前 thread | `tl.program_id(0)` 选出当前 program |
| 一次分多少数据 | 通常从一个 thread 的标量工作开始 | 一个 program 的 tile |
| 生成下标 | 标量 `index`，或在线程循环中递增 | 张量 `offsets` |
| 保护尾部 | `if (index < size)` | `mask = offsets < size` |
| 读取 | 指针下标或显式向量类型 | `tl.load(pointer + offsets, mask=...)` |
| 计算 | C++ 标量/向量表达式 | tile 上的张量表达式 |
| 写回 | 指针下标 | `tl.store(..., mask=...)` |
| 主要显式参数 | grid、block、每线程工作、加载类型 | program grid、block size、num warps |
| 主要风险 | 越界、对齐、并行度、资源压力 | mask、tile 过大、program 映射、资源压力 |

HIP 的一个 thread 与 Triton 的一个 program **不是一一对应**。更准确的迁移方法是先问三次：

```text
1. 这一组输出下标是什么？
2. 它们读取哪些输入下标？
3. 尾部哪些位置必须关闭？
```

只要这三件事对应起来，语言差异就不会遮住算子本身。


### 8.7.2 什么时候先选哪条路线

| 当前目标 | 更自然的起点 | 原因 |
| ---- | ---- | ---- |
| 第一次验证一个简单算子想法 | Triton | kernel 与 Host wrapper 较短，tile/mask 容易修改 |
| 精确控制 thread 到地址的排列 | HIP | grid、thread、指针与加载类型直接暴露 |
| 研究 Wave/LDS/VGPR 细节 | HIP | 更靠近 AMD 执行与资源模型 |
| 快速尝试多个 tile 参数 | Triton | meta-parameter 与 Python 驱动更集中 |
| 定位地址或 mask 错误 | Triton + Triton-viz，或 HIP + trace | 两边都要回到实际地址证据 |

同一个算子可以先用 Triton 验证算法，再用 HIP 追底层细节；也可以从 HIP 基准实现出发，完全不重写。路线选择服务于当前问题，不建立跨 shape、跨软件栈的语言排名。


### 8.7.3 怎样读这次对照

这次 HIP 与 Triton 对照使用相同输入公式、FP32 语义、shape、warmup、repeat 与 kernel-only 计时边界。当前快照中 `triton-t0` 与 `hip-v0` 的三进程范围重叠，中心值差约 `0.00012 ms`；这组数据只适用于当前参考平台，不用于推断另一种环境中的胜负。

更可靠的共同结论是：先保持地址连续和边界正确，再用目标 shape 实测 grid、tile 与向量类型；更少的 block/program 或更宽的源码类型都不自动等于更快。


## 8.8 负结果、适用边界与下一步

以下固定数值来自 gfx1201 reference；本次平台应以上方动态 evidence 为准。负结果不是删掉的草稿，而是下一轮实验的输入：

- 受控跨步版本的 median 是 `2.567170 ms`，逻辑有效带宽是 `78.423556 GB/s`；它是本次最明确的负例，但仍没有直接测得物理显存事务。
- `hip-v2` 把 block 数从 65536 限制为 256，median 为 `0.343484 ms`，没有超过 `hip-v0` 的 `0.336324 ms`。减少 grid 不自动带来收益。
- `hip-v3` 与 `hip-v2` 使用相同的 256 blocks，但 `float4` 版本 median 为 `0.345224 ms`；源码向量类型没有在当前配置下提速。
- `triton-t1` 把 program 数从 65536 减到 16384，median 为 `0.338504 ms`，没有超过 `triton-t0` 的 `0.336204 ms`；trace 同时显示 VGPR 从 8 变为 24。

这些结论只适用于 manifest 记录的硬件、软件、shape、block、warmup、repeat 与独立进程协议。下一步若要检验可迁移性，应先扩展 shape、dtype 或系统状态中的一个变量，并生成新的 manifest 与 curated evidence；不要把当前逻辑有效带宽解释成物理 GDDR6 流量。


## 8.9 复跑、练习与验收


### 8.9.1 Notebook 默认完整流程与手工入口

Notebook 从头 Run All 时默认执行 `run_all.sh → profile_all.sh` 链路；Triton profiling 默认自动选择：检测到 Python 环境自带配套 ROCm SDK（`_rocm_sdk_core` 内含 rocprofv3）时用 `direct` 并采集成完整 evidence，否则回退到安全的 `skip` 模式。整体在临时副本中运行，并把本次产物按架构和 UTC run id 隔离保存：

```text
runs/chapter8/<arch>/<UTC-run-id>/
├── evidence/
├── logs/
├── profiles/
└── notebook_status.json
```

该路径不会覆盖 `code/part2-kernels/chapter8/evidence/` 中已提交的 gfx1201 reference。缺少 Triton-viz 时记录 SKIP 并继续；benchmark 成功后仍会调用 `profile_all.sh`，若缺少 rocprofv3，该阶段会明确失败并使 `CH8_PROFILE_COMPLETE` 与 `CH8_FORMAL_COMPLETE` 保持 False。

若需要在终端手工刷新 curated evidence，只能在可随时丢弃的临时仓库副本中运行下列命令；它们会直接改写 Chapter 8 的 `evidence/`、`logs/` 与 `profiles/`：

```bash
cd code/part2-kernels
# SOURCE_COMMIT 必须是当前 HEAD（完整 SHA 或合法前缀）；无 git 时人工核对后填入。
GPU_ARCH=<gfx1100|gfx1151|gfx1201> HELLO_GPU_ARCH=<same-target> SOURCE_COMMIT=<verified-sha> \
  timeout 900s bash chapter8/run_all.sh
# 仅在 run_all.sh 成功完成并产生 benchmark 产物后执行：
GPU_ARCH=<gfx1100|gfx1151|gfx1201> HELLO_GPU_ARCH=<same-target> SOURCE_COMMIT=<verified-sha> \
  TRITON_PROFILE_MODE=direct timeout 300s bash chapter8/profile_all.sh
```

上面的 `direct` 只用于已经通过独立 rocprofv3 + Triton 测试的兼容平台；未设置时自动选择（环境自带配套 ROCm SDK 时 `direct`，否则 `skip`）。

`run_all.sh` 的顺序是：

```text
采集环境
→ 编译 HIP
→ HIP/Triton 边界正确性
→ 可用时执行 Triton-viz 检查
→ 主 benchmark
→ 3 次独立进程复跑
→ 汇总 CSV/JSON
```

`profile_all.sh` 单独运行，避免 profiler 开销混入 GPU event benchmark。现有绘图脚本仍读取指定 evidence：

```bash
python chapter8/plot_vector_add_ch7.py \
  --summary chapter8/evidence/summary.csv \
  --manifest chapter8/evidence/manifest.json \
  --out ../../docs/part2-kernels/chapter8/images/vector-add-ch7-bandwidth.png
```


### 8.9.2 从 Add 迁移到更多逐元素算子

Vector Add 的价值不在于加法本身，而在于它提供了一个可替换的模板。

把核心表达式改成 ReLU：

```text
output[i] = max(input[i], 0)
```

线程/program 的划分与尾部保护可以保持不变。改成 Scale & Bias：

```text
output[i] = alpha * input[i] + beta
```

仍然是逐元素，只是每个位置多做了乘法和加法。把 Add 与 ReLU 合在一个 kernel：

```text
output[i] = max(input_a[i] + input_b[i], 0)
```

如果分成两个 kernel，中间结果通常需要写回再读出；融合后可能减少这次中间读写。这里仍然只能提出假设，真正的收益要在第 12 章用完整计时验证。


### 8.9.3 练习

1. 把 `N` 改成 `1、31、32、33、1027`，先预测 HIP 的 `if` 与 Triton 的 mask 分别关闭哪些位置，再运行正确性检查。
2. 把 Triton t1 的 `BLOCK_SIZE` 改为 `512`，保持 `num_warps=4`，记录 program 数量怎样变化；不要在计时前猜谁更快。
3. 在 HIP 与 Triton 中都实现 `Scale & Bias`，继续使用同一输入生成、precheck、postcheck 与 event 计时框架。
4. 给 `float4` 版本传入从 `input + 1` 开始的偏移指针，先解释为什么对齐假设被破坏，再设计安全的头部/主体/尾部拆分；不要直接运行未对齐强转。


### 8.9.4 验收信号

完成本章时，不要求某个版本必须最快，但需要同时满足下面四项：

1. HIP 与 Triton 的边界正确性检查都通过，发布行的 `max_abs_error` 与 evidence 一致。
2. 能解释 benchmark 的 shape、warmup、repeat、独立进程汇总和 kernel-only 计时范围。
3. 能从 profile 表指出受控 HIP 对照固定了哪些资源字段，以及 Triton tile 变化伴随什么资源变化。
4. 能明确写出至少一个负结果，并说明当前结论仅适用于记录中的硬件、shape 与物理流量口径。


---

## Expected output / interpretation

> 下列固定性能数字均为 gfx1201 reference。当前平台的验收以 8.6 动态读取的本次 evidence 和四个完成状态为准。

### 正确性

- 所有实现的 `correct` 字段应为 `OK`
- `max_abs_error` 应为 `0.0`（FP32 精确加法）
- 边界测试（N=31, 32, 33, 1027）全部通过

### 性能趋势

1. **hip-v1-strided 显著慢于 hip-v1-contiguous**
   - 参考数据：约 6.95× 倍
   - 原因：跨步访问破坏了地址连续性
   - 两者的 trace 资源字段完全相同，差异来自访存模式

2. **hip-v0 与 triton-t0 性能接近**
   - 两者都是最简单的正确实现
   - 中位数差异约 0.00012 ms（在三进程范围内）

3. **负结果**
   - hip-v2 (Grid-Stride) 未超过 hip-v0
   - hip-v3 (float4) 未超过 hip-v2
   - triton-t1 (BLOCK_SIZE=1024) 未超过 triton-t0
   - 这些都是有效结果，说明更少 block 或源码向量化不自动带来收益

### Profiling 解读

- **hip-v1 受控对照**：contiguous 与 strided 的 Grid、Workgroup、VGPR、SGPR、LDS、Scratch 完全相同，性能差异纯粹来自地址顺序
- **triton-t1 资源变化**：VGPR 从 8 增加到 24，可能影响占用率
- **LDS 与 Scratch**：所有实现均为 0，本章算子不需要共享内存或寄存器溢出

### 有效带宽

- 逻辑有效带宽 = (3 × N × 4 Byte) / (median_ms × 1e-3) / 1e9
- hip-v0 约 598.6 GB/s（gfx1201 参考平台）
- hip-v1-strided 约 78.4 GB/s（受控负例）
- **不等于物理 GDDR6 流量**，需要硬件计数器进一步验证

---

## Pass criteria

完成本章操作后，你应该能够：

1. **正确性**
   - ✓ 所有实现通过边界测试（N=31, 32, 33, 1027）
   - ✓ `max_abs_error = 0.0`
   - ✓ precheck 与 postcheck 均为 OK

2. **性能理解**
   - ✓ 能解释 hip-v1-strided 为何显著慢于 hip-v1-contiguous
   - ✓ 能说明逻辑有效带宽的计算公式与适用范围
   - ✓ 能识别负结果（Grid-Stride、float4、更大 tile 未提速）

3. **Profiling 分析**
   - ✓ 能从 profile_summary.csv 指出受控对照固定了哪些资源字段
   - ✓ 能解释 triton-t1 的 VGPR 变化
   - ✓ 能说明为什么 LDS 和 Scratch 为 0

4. **实验设计**
   - ✓ 能说明为什么 hip-v1 受控对照只改变地址顺序
   - ✓ 能解释为什么需要独立进程复跑（INDEPENDENT_RUNS=3）
   - ✓ 能说明当前结论仅适用于记录中的硬件或 shape

5. **HIP vs Triton**
   - ✓ 能用表格对比两种范式的关键差异（thread vs program, 标量 vs tile）
   - ✓ 能解释 Triton 的 mask 如何保护尾部
   - ✓ 能说明 `blockDim.x` 与 `BLOCK_SIZE` 不是对应参数

---

## 本章小结

- Element-Wise 的核心不是“公式简单”，而是输出位置之间没有依赖，可以独立划分。
- HIP 线程级范式让 kernel 正文从一个 thread 与标量下标出发；Triton 分块范式让正文从一个 program 与一块逻辑下标出发。`blockDim.x` 与 `BLOCK_SIZE` 不是对应参数，tile 位置也不固定对应硬件 lane。
- Vector Add 每个 FP32 输出至少对应两次逻辑读取、一次逻辑写回和一次加法；受控跨步实验让用时增加到连续版的约 `6.95×`，结果与数据搬运主导假设一致。
- HIP 从 thread 与标量地址出发，适合看清连续访问、Grid-Stride、向量类型和尾部。
- Triton 从 program 与 tile offsets 出发，用 mask 统一处理边界；Triton-viz 可以把地址访问展开，但不代替 GPU 性能实验。
- Grid-Stride 在当前配置下与 v0 接近；`float4` 没有提速；Triton 的 1024 元素 tile 也没有稳定超过 256 元素 tile。负结果同样决定下一轮该测什么。
- 遇到一个新的逐元素算子时，本章的判断仍然成立：先确认输出位置彼此独立 → 沿用同一套线程/program 划分与尾部保护 → 只替换核心表达式 → 用同一口径复测，不要凭源码表象预判快慢。
- 下一章会撤掉“每个输出彼此独立”这个前提：Reduction 需要许多线程合作得到一个结果，因此会第一次引入跨线程通信、LDS 与 Wave Shuffle。


## 延伸阅读

- [AMD HIP 编程模型](https://rocm.docs.amd.com/projects/HIP/en/latest/understand/programming_model.html)：thread、block、grid、wavefront 与 SIMT 执行关系。
- [AMD HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)：线程映射、合并访存、对齐与内存吞吐建议。
- [Triton 官方编程指南：Introduction](https://triton-lang.org/main/programming-guide/chapter-1/introduction.html)：Blocked Program、Scalar Threads 与分块算法的官方定义。
- [Triton 官方 Vector Addition 教程](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html)：kernel、Host wrapper、正确性与 benchmark 的官方最小例子。
- [Triton 官方调试文档](https://triton-lang.org/main/programming-guide/chapter-3/debugging.html)：CPU interpreter 与 Triton-viz 的定位。
- [Triton-viz 官方仓库](https://github.com/Deep-Learning-Profiling-Tools/triton-viz)：trace、可视化、profiler 与 sanitizer 的使用入口。
- [PyTorch HIP 语义](https://docs.pytorch.org/docs/stable/notes/hip.html)：为什么 ROCm 构建继续使用 `torch.cuda` 接口名。
- [DLog Element-Wise 教程](https://dlog.com.cn/posts/cuda05/element_wise)：本文只借鉴“先画数据移动、再进入代码”的教学节奏，图与实验均按 AMD/ROCm 语境重新制作。


## 优化效果总览（折线图）

把 7 个实现放到同一张折线图：横轴按 `hip-v0 → hip-v1-contiguous → hip-v1-strided → hip-v2 → hip-v3 → triton-t0 → triton-t1` 排列，纵轴为逻辑有效带宽（GB/s）。蓝线为本平台实测，灰虚线为 gfx1201 reference；点上标注带宽数值，图注列出本平台各实现相对 `v0` 的带宽倍率。

读图要点：

1. **v1-contiguous vs v0**：合并访存让带宽贴近峰值，是本章第一个正向优化；
2. **v1-strided**：受控负例，跨步访问让带宽跌到谷底，说明瓶颈在访存模式而非计算；
3. **v2 / v3 / triton-t0 / triton-t1**：在连续访问前提下基本贴住带宽上限，印证 Element-Wise 的性能天花板由内存带宽决定；
4. 若本次未产生当前平台 evidence，图中只保留 reference 曲线。

> 该图由 `code/part2-kernels/chapter8/plot_optimization_line.py` 生成，数据来自本次运行与 reference 的 `evidence/summary.csv`。

In [ ]:
# 优化效果总览：折线图（当前平台 vs gfx1201 reference）
import importlib.util

plot_script = code_dir / "plot_optimization_line.py"
if not plot_script.is_file():
    print(f"缺少绘图脚本: {plot_script}")
elif importlib.util.find_spec("matplotlib") is None:
    print("缺少 matplotlib，跳过优化效果折线图；可先安装：")
    print(f"  {sys.executable} -m pip install matplotlib")
else:
    viz_root = (
        CURRENT_RUN_DIR
        if CURRENT_RUN_DIR is not None
        else repo_root / "runs" / "chapter8" / arch
    ) / "viz"
    viz_root.mkdir(parents=True, exist_ok=True)
    out_png = viz_root / "optimization-linechart.png"
    command = [
        sys.executable,
        str(plot_script),
        "--out",
        str(out_png),
    ]
    if (
        REFERENCE_EVIDENCE_DIR is not None
        and (REFERENCE_EVIDENCE_DIR / "summary.csv").is_file()
    ):
        command += [
            "--reference-summary",
            str(REFERENCE_EVIDENCE_DIR / "summary.csv"),
            "--reference-manifest",
            str(REFERENCE_EVIDENCE_DIR / "manifest.json"),
        ]
    if (
        CURRENT_EVIDENCE_DIR is not None
        and (CURRENT_EVIDENCE_DIR / "summary.csv").is_file()
    ):
        command += [
            "--current-summary",
            str(CURRENT_EVIDENCE_DIR / "summary.csv"),
            "--current-manifest",
            str(CURRENT_EVIDENCE_DIR / "manifest.json"),
        ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0 and out_png.is_file():
        from IPython.display import Image, display

        display(Image(filename=str(out_png)))
        print(f"折线图已保存: {out_png}")
    else:
        print("折线图生成失败：")
        print(result.stdout)
        print(result.stderr)